# 랭체인의 메시지 히스토리

- messages 리스트를 직접 관리하는 대신, 랭체인이 '세션(session_id) 별 대화 기록'을 자동으로 보관하게 만든다.
    - `InMemoryChatMessageHistory` : 메모리에 대화 기록을 저장하는 객체
    - `RunnableWithMessageHistory` : 모델 실행 시 그 기록을 자동으로 끼워주는 래퍼
- session_id 가 다르면 서로 다른 대화로 취급된다(맥락이 분리됨).

> 참고: `RunnableWithMessageHistory` 는 최신 랭체인에서 deprecated 경고가 날 수 있다(LangGraph 메모리로 대체되는 추세). 책의 흐름을 따라 그대로 사용한다.

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory  # 메모리에 대화 기록을 저장하는 클래스
from langchain_core.runnables.history import RunnableWithMessageHistory  # 메시지 기록을 활용해 실행 가능한 래퍼
from langchain_openai import ChatOpenAI  # 오픈AI 모델을 사용하는 랭체인 챗봇 클래스
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
import os

load_dotenv()

model = ChatOpenAI(
    model="openai/gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",  # 필수!
    api_key=os.getenv('OPENAI_API_KEY'),       # OpenRouter 전용 키
)

# 세션별 대화 기록을 저장할 딕셔너리 {session_id: 대화기록객체}
store = {}

In [ ]:
# 세션 ID에 따라 대화 기록을 가져오는 함수
def get_session_history(session_id: str):
    # 만약 해당 세션 ID가 store에 없으면, 새로 생성해 추가함
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()  # 메모리에 대화 기록을 저장하는 객체 생성
    return store[session_id]  # 해당 세션의 대화 기록을 반환

# 모델 실행 시 대화 기록을 함께 전달하는 래퍼 객체 생성
with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [ ]:
config = {"configurable": {"session_id": "abc2"}}  # 세션 ID를 설정하는 config 객체 생성

response = with_message_history.invoke(
    [HumanMessage(content="안녕? 난 이병일이야.")],
    config=config,
)

print(response.content)

In [ ]:
# 같은 세션(abc2)이라 이름을 기억한다.
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

print(response.content)

In [ ]:
# 세션 ID를 abc3 으로 바꾸면 맥락이 분리되어 이름을 모른다.
config = {"configurable": {"session_id": "abc3"}}

response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,
)

response.content

In [ ]:
# 다시 abc2 세션으로 돌아오면 이전 대화를 기억한다.
config = {"configurable": {"session_id": "abc2"}}

response = with_message_history.invoke(
    [HumanMessage(content="아까 우리가 무슨 얘기 했지?")],
    config=config,
)

response.content

## 스트림 방식으로 출력하기

`stream()` 으로 답변을 조각(chunk) 단위로 받아 출력한다.

In [ ]:
config = {"configurable": {"session_id": "abc2"}}
for r in with_message_history.stream(
    [HumanMessage(content="내가 어느 나라 사람인지 맞춰보고, 그 나라의 문화에 대해 말해봐")],
    config=config,
):
    print(r.content, end="|")  # 조각 경계를 | 로 표시